## initial installing commands





In [ ]:
# !pip install "alphatims[plotting]"

# !pip install hvplot

# !pip install jupyter_bokeh

# !pip install dask[dataframe]

# !pip install matplotlib

# !pip install scikit-learn

In [ ]:
import alphatims.utils
import alphatims.bruker
import alphatims.plotting
import importlib
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN

from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
import seaborn as sns



In [ ]:


alphatims.utils.set_threads(4)
log_file_name = alphatims.utils.set_logger(
    log_file_name="tutorial_log.txt",
    overwrite=True
)

def reload():
    importlib.reload(alphatims.utils)
    importlib.reload(alphatims.bruker)
    importlib.reload(alphatims.plotting)
    alphatims.utils.set_threads(4)
    alphatims.utils.set_logger(log_file_name="tutorial_log.txt")
    
isDia = True

if (isDia):
    # bruker_d_folder_name = "../MS_Data/P4139_02_1_1_509.hdf"
    # bruker_d_folder_name = "../MS_Data/20201207_tims03_Evo03_PS_SA_HeLa_200ng_EvoSep_prot_high_speed_21min_8cm_S1-C8_1_22474.hdf"
    bruker_d_folder_name = "../MS_Data/test1.hdf"
    
else :
    bruker_d_folder_name = ""

data = alphatims.bruker.TimsTOF(bruker_d_folder_name)

## Selecting the precursor in the fist quardupole window

### Extrating precursor detector events in the first Quadrupole window, for a given rt range and mobility range

In [ ]:
def extractPrecursorAndFragment(quad_window_index = 1,mz_range_start=400.0,quad_window_size=25.0 ):
    # first windows index is 1
    maximum_precursor_indices = 8
    window_start_mz = mz_range_start + (quad_window_index-1)* quad_window_size
    window_end_mz = window_start_mz + quad_window_size
    # window_1_precursor = data[:,:,0,window_start_mz: window_end_mz]


    selected_mobility = 0.765332
    mobility_threshold = 0.06


    selected_rt_value = 141.751394
    rt_value_threshold = 60


    selected_fragments= data[
        selected_rt_value - rt_value_threshold/2: selected_rt_value + rt_value_threshold/2,
        selected_mobility - mobility_threshold/2: selected_mobility + mobility_threshold/2,
        ((quad_window_index -1) % maximum_precursor_indices) +1,
        :
    ]
    # following may not nesessory because of the selection use rt and mobility 
    selected_fragments = selected_fragments[selected_fragments["quad_low_mz_values"]== window_start_mz]


    selected_precursors = data[
        :,
        # selected_rt_value - rt_value_threshold/2: selected_rt_value + rt_value_threshold/2,
        :,
        # selected_mobility - mobility_threshold/2: selected_mobility + mobility_threshold/2,
        0,
        window_start_mz:window_end_mz,
        # 'raw'
    ]

    return selected_precursors, selected_fragments


In [ ]:
# need to refactor the extractPrecursorAndFragment function
q1_selected_precursors , selected_fragments = extractPrecursorAndFragment(quad_window_index=1)
q1_selected_precursors

## Peak detection 

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['agg.path.chunksize'] = 10000  # Increase this value as needed


def plot_mz_intensity(df1, df2=None, label1="Dataset 1", label2="Dataset 2", color1='black', color2='red'):
    """
    Plots m/z values against intensity values for one or two datasets.
    
    Parameters:
        df1 (DataFrame): First dataset with 'mz_values' and 'intensity_values'.
        df2 (DataFrame, optional): Second dataset with 'mz_values' and 'intensity_values'.
        label1 (str): Label for the first dataset.
        label2 (str): Label for the second dataset.
        color1 (str): Color for the first dataset.
        color2 (str): Color for the second dataset.
    """
    plt.figure(figsize=(10, 5))
    
    # Plot the first dataset
    plt.plot(df1['mz_values'], df1['intensity_values'], label=label1, color=color1, alpha=0.6)
    
    # Plot the second dataset if provided
    if df2 is not None:
        plt.plot(df2['mz_values'], df2['intensity_values'], label=label2, color=color2, alpha=0.6)
    
    plt.xlabel("m/z")
    plt.ylabel("Intensity")
    plt.title("m/z vs Intensity Plot")
    plt.legend()
    plt.show()





In [ ]:
def plot_lineplot(x_dim,y_dim,selected_indices):
        if data is not None:  # Ensure dataset is loaded
            # Validate selected dimensions
            valid_x_dims = ['mz', 'rt', 'mobility']
            valid_y_dims = ['intensity', 'mz', 'rt', 'mobility']
            if x_dim in valid_x_dims and y_dim in valid_y_dims: 
                # Create the plot
                plot = alphatims.plotting.line_plot(
                    data,
                    selected_indices,
                    x_dim,
                    data.sample_name,
                    y_dim
                )
                print("acquisition_mode: ",data.acquisition_mode)
                return plot  # Dynamically update the plot
            else:
                return "Invalid dimensions selected."
        else:
            return "Dataset is not loaded."

In [ ]:
selected_precursors = data[:,:,0,400.0:425.0,"raw"]


In [ ]:
# plot_lineplot('mz','intensity',selected_precursors)

In [ ]:
import numpy as np
import pandas as pd


# finding the similar raws with the selected thresholds 
def group_similar_rows(df, rt_tolerance, mobility_tolerance, mz_tolerance):
    min_values = df.min()
    max_values = df.max()
    
    rt_min_value = min_values['rt_values']
    rt_max_value = max_values['rt_values']
    
    mobility_min_value = min_values['mobility_values']
    mobility_max_value = max_values['mobility_values']
    
    mz_min_values = min_values['mz_values']
    mz_max_values = max_values['mz_values']
    
    
    df = df.sort_values(by=["rt_values"]).reset_index(drop=True)
    rt_bins = np.arange(rt_min_value, rt_max_value + rt_tolerance, rt_tolerance)
    df["rt_group"] = pd.cut(df["rt_values"], rt_bins,  labels=rt_bins[:-1])
    
    
    df = df.sort_values(by=["mobility_values"]).reset_index(drop=True)
    mobility_bins = np.arange(mobility_min_value, mobility_max_value + mobility_tolerance, mobility_tolerance)
    df["mobility_group"] = pd.cut(df["mobility_values"], mobility_bins,  labels=mobility_bins[:-1])
    
    
    df = df.sort_values(by=["mz_values"]).reset_index(drop=True)
    mobility_bins = np.arange(mz_min_values, mz_max_values +mz_tolerance, mz_tolerance)
    df["mz_group"] = pd.cut(df["mz_values"], mobility_bins,  labels=mobility_bins[:-1])
    
    

    grouped_df = df.groupby(["rt_group", "mobility_group", "mz_group"], observed=True).agg({
        "raw_indices": "first",
        "rt_values": "mean",
        "mobility_values": "mean",
        "mz_values": "mean",
        "intensity_values": "sum"
    }).reset_index(drop=False)
    return grouped_df

In [ ]:
maxId = q1_selected_precursors['intensity_values'].idxmax()
working_df = q1_selected_precursors.copy()
peak = working_df.loc[maxId]
rt_tolerance =  6
mobility_tolerance = 0.02
mz_tolerance = 0.02
# Create proximity mask

print()
mask = (
    working_df['rt_values'].between(peak['rt_values'] - rt_tolerance, 
                                    peak['rt_values'] + rt_tolerance) &
    working_df['mobility_values'].between(peak['mobility_values'] - mobility_tolerance,
                                            peak['mobility_values'] + mobility_tolerance) &
    working_df['mz_values'].between(peak['mz_values'] - mz_tolerance,
                                    peak['mz_values'] + mz_tolerance)
)
mask
# q1_selected_precursors.loc[maxId]['intensity_values'] ==q1_selected_precursors['intensity_values'].max()


# q1_selected_precursors[q1_selected_precursors['intensity_values'] == 1.574500e+04]

In [ ]:
# import numpy as np
# import pandas as pd

# def group_similar_rows_1(df, rt_tolerance, mobility_tolerance, mz_tolerance):
#     working_df = df.copy()
#     result_rows = []
    
#     while not working_df.empty:
#         # Find peak with highest intensity
#         peak_idx = working_df['intensity_values'].idxmax()
#         peak = working_df.loc[peak_idx]
        
#         # Create proximity mask
#         mask = (
#             working_df['rt_values'].between(peak['rt_values'] - rt_tolerance, 
#                                            peak['rt_values'] + rt_tolerance) &
#             working_df['mobility_values'].between(peak['mobility_values'] - mobility_tolerance,
#                                                  peak['mobility_values'] + mobility_tolerance) &
#             working_df['mz_values'].between(peak['mz_values'] - mz_tolerance,
#                                            peak['mz_values'] + mz_tolerance)
#         )
        
        
        
#         # Aggregate group
#         group = working_df[mask]
#         aggregated = {
#             'raw_indices': group['raw_indices'].iloc[0],
#             'rt_values': group['rt_values'].mean(),
#             'mobility_values': group['mobility_values'].mean(),
#             'mz_values': group['mz_values'].mean(),
#             'intensity_values': group['intensity_values'].sum()
#         }
#         result_rows.append(aggregated)
        
#         # Remove processed rows
#         working_df = working_df[~mask]
    
#     return pd.DataFrame(result_rows)
import numpy as np
import pandas as pd

def group_similar_rows_optimized(df, rt_tolerance, mobility_tolerance, mz_tolerance):
    # Prepare a copy of the DataFrame to work on
    working_df = df.copy()
    result_rows = []
    
    # Convert DataFrame columns to NumPy arrays for faster computation
    rt_values = working_df['rt_values'].values
    mobility_values = working_df['mobility_values'].values
    mz_values = working_df['mz_values'].values
    intensity_values = working_df['intensity_values'].values
    
    while len(working_df) > 0:
        # Find index of peak with highest intensity
        peak_idx = np.argmax(intensity_values)
        
        # Extract peak values
        peak_rt = rt_values[peak_idx]
        peak_mobility = mobility_values[peak_idx]
        peak_mz = mz_values[peak_idx]
        
        # Create proximity mask using vectorized operations
        mask = (
            (np.abs(rt_values - peak_rt) <= rt_tolerance) &
            (np.abs(mobility_values - peak_mobility) <= mobility_tolerance) &
            (np.abs(mz_values - peak_mz) <= mz_tolerance)
        )
        
        # Aggregate group using masked indices
        group_indices = np.where(mask)[0]
        aggregated = {
            'raw_indices': working_df.iloc[group_indices[0]]['raw_indices'],
            'rt_values': rt_values[mask].mean(),
            'mobility_values': mobility_values[mask].mean(),
            'mz_values': mz_values[mask].mean(),
            'intensity_values': intensity_values[mask].sum()
        }
        result_rows.append(aggregated)
        
        # Remove processed rows from arrays and DataFrame
        mask_inverse = ~mask
        rt_values = rt_values[mask_inverse]
        mobility_values = mobility_values[mask_inverse]
        mz_values = mz_values[mask_inverse]
        intensity_values = intensity_values[mask_inverse]
        working_df = working_df.iloc[np.where(mask_inverse)[0]]
    
    return pd.DataFrame(result_rows)


In [ ]:
group_similar_rows_optimized(
    q1_selected_precursors, 
    rt_tolerance = 6, 
    mobility_tolerance=0.005, 
    mz_tolerance= 0.025
    )

In [ ]:
grouped_window_1_precursors =  group_similar_rows(
    q1_selected_precursors, 
    rt_tolerance= 6, 
    mobility_tolerance=0.005, 
    mz_tolerance= 0.025
    ).sort_values(by="mz_values",ascending= True)


In [ ]:
plot_mz_intensity(grouped_window_1_precursors)

get the higest peak

In [ ]:
# aggregated = q1_selected_precursors.groupby(["frame_indices", "tof_indices", "scan_indices"], observed=True).agg({
aggregated = q1_selected_precursors.groupby(["frame_indices", "tof_indices"], observed=True).agg({
        "raw_indices": "first",
        "rt_values": "first",
        "mobility_values": "mean",
        "mz_values": "first",
        "intensity_values": "sum"
    }).reset_index(drop=False)
aggregated

In [ ]:
plot_mz_intensity(aggregated)

In [ ]:
plot_mz_intensity(aggregated)

In [ ]:
plot_mz_intensity(grouped)

In [ ]:
grouped = group_similar_rows(
    aggregated, 
    rt_tolerance= 30, 
    mobility_tolerance=0.005, 
    mz_tolerance= 0.025
    ).sort_values(by="mz_values",ascending= True)
grouped

In [ ]:
hightest_peaks = grouped_window_1_precursors[grouped_window_1_precursors['intensity_values'] == grouped_window_1_precursors['intensity_values'].max()]
hightest_peaks



finding adjecent peaks for the highest peak

In [ ]:
def findAdjecentPeakForThePrecursor( index_location, peaks_in_selected_precursors, precursorDf, ms_diff ):
    # index_location = 0
    # ms_diff = charge_1_ms_diff
    selected_peak = peaks_in_selected_precursors.iloc[index_location]
    mz_value = selected_peak['mz_values']
    intensity_value = selected_peak['intensity_values']
    filtered_raws_1 = precursorDf[precursorDf['mz_values']== mz_value]
    # filtered_raws_1
    isotop_arr = []
    for i in range(7):
        error_tolerance = 0.01
        filtered_raws_1 = precursorDf[(precursorDf['mz_values'] >= mz_value + (i-3)*ms_diff - error_tolerance) & ( precursorDf['mz_values'] <= mz_value + (i-3)*ms_diff+error_tolerance)]
        # filtered_raws_1 = precursorDf[precursorDf['mz_values']== mz_value + (i-3)*ms_diff]
        isotop_arr.append(not filtered_raws_1.empty)
    return isotop_arr


In [ ]:
for index, row in hightest_peaks.iterrows():
    selected_mobility = row['mobility_values']
    selected_rt_value = row['rt_values']
    selected_mz_value = row['mz_values']
    selected_intensity_value = row['intensity_values']
    selected_raw_indices = row['raw_indices']
    
    print(f"index : {index}")
    print(f"Selected Mobility: {selected_mobility}")
    print(f"Selected RT Value: {selected_rt_value}")
    print(f"Selected m/z Value: {selected_mz_value}")
    print(f"Selected Intensity Value: {selected_intensity_value}")
    print(f"Selected Raw Indices: {selected_raw_indices}")
    
    charge_ms_diff = {
        1: 1.003,
        2: 0.5015,
        3: 0.3343,
        4: 0.2508
    }
    for key, ms_diff in charge_ms_diff.items():
        isotop_arr = []
        for i in range(7):
            error_tolerance = 0.01
            print(i)
            filtered_raws_1 = selected_precursors[(selected_precursors['mz_values'] >= selected_mz_value + (i-3)*ms_diff - error_tolerance) & ( selected_precursors['mz_values'] <= selected_mz_value + (i-3)*ms_diff+error_tolerance)]
            print(filtered_raws_1)
            break
            # filtered_raws_1 = precursorDf[precursorDf['mz_values']== mz_value + (i-3)*ms_diff]
            isotop_arr.append(not filtered_raws_1.empty)    
        print(isotop_arr)
    break

In [ ]:
filtered_raws_1

In [ ]:
def detect_peaks(selected_precursors, window_length=11, polyorder=3, intensity_percentile=99):
    """
    Detects peaks in mass spectrometry data using the Savitzky-Golay filter.
    
    Parameters:
    - selected_precursors (pd.DataFrame): DataFrame with 'mz_values' and 'intensity_values'.
    - window_length (int): Window size for the Savitzky-Golay filter (default: 11).
    - polyorder (int): Polynomial order for the filter (default: 3).
    - intensity_percentile (float): Percentile threshold for peak selection (default: 99).

    Returns:
    - peak_df (pd.DataFrame): DataFrame of detected peaks.
    - sorted_selected_precursors (pd.DataFrame): Input data with first derivative added.
    """
    # Sort by m/z values
    sorted_selected_precursors = selected_precursors.sort_values(by="mz_values").copy()

    # Apply Savitzky-Golay filter to get first derivative
    first_derivative = savgol_filter(sorted_selected_precursors['intensity_values'], window_length, polyorder, deriv=1)
    sorted_selected_precursors['first_derivation'] = first_derivative

    # Find zero crossings (potential peak locations)
    zero_crossings = np.where(np.diff(np.sign(first_derivative)) < 0)[0]

    # Apply intensity threshold to filter out noise
    intensity_threshold = np.percentile(sorted_selected_precursors['intensity_values'], intensity_percentile)
    peaks = [i for i in zero_crossings if sorted_selected_precursors['intensity_values'].iloc[i] > intensity_threshold]

    # Extract detected peaks
    peak_df = sorted_selected_precursors.iloc[peaks]

    # Plot results
    plt.figure(figsize=(10, 5))
    plt.plot(sorted_selected_precursors['mz_values'], sorted_selected_precursors['intensity_values'], label="Raw Data", color='black', alpha=0.6)
    plt.scatter(peak_df['mz_values'], peak_df['intensity_values'], color='red', label="Detected Peaks", zorder=3)
    plt.xlabel("m/z")
    plt.ylabel("Intensity")
    plt.title("Peak Detection Using Savitzky-Golay Derivatives")
    plt.legend()
    plt.show()

    return peak_df, sorted_selected_precursors

In [ ]:
peak_df, sorted_selected_precursors = detect_peaks(selected_precursors)
peak_df = peak_df.sort_values(by="intensity_values", ascending=False)
 

In [ ]:
higest_peak = peak_df.iloc[0]
higest_peak

### Isotopes Identification to find the Charge of precursor peptides

In [ ]:
charge_1_ms_diff = 1.003
charge_2_ms_diff = 0.5015
charge_3_ms_diff = 0.3343
charge_4_ms_diff = 0.2508

In [ ]:
def findAdjecentPeakForThePrecursor( index_location, peaks_in_selected_precursors, precursorDf, ms_diff ):
    # index_location = 0
    # ms_diff = charge_1_ms_diff
    selected_peak = peaks_in_selected_precursors.iloc[index_location]
    mz_value = selected_peak['mz_values']
    intensity_value = selected_peak['intensity_values']
    filtered_raws_1 = precursorDf[precursorDf['mz_values']== mz_value]
    # filtered_raws_1
    isotop_arr = []
    for i in range(7):
        error_tolerance = 0.01
        filtered_raws_1 = precursorDf[(precursorDf['mz_values'] >= mz_value + (i-3)*ms_diff - error_tolerance) & ( precursorDf['mz_values'] <= mz_value + (i-3)*ms_diff+error_tolerance)]
        # filtered_raws_1 = precursorDf[precursorDf['mz_values']== mz_value + (i-3)*ms_diff]
        isotop_arr.append(not filtered_raws_1.empty)
    return isotop_arr


In [ ]:
# index_location = 0
# ms_diff = charge_1_ms_diff
# selected_peak = peaks_in_selected_precursors.iloc[index_location]
# mz_value = selected_peak['mz_values']
# intensity_value = selected_peak['intensity_values']
# filtered_raws_1 = selected_precursors[selected_precursors['mz_values']== mz_value]
# filtered_raws_1
# isotop_arr = []
# for i in range(7):
#     filtered_raws_1 = selected_precursors[selected_precursors['mz_values']== mz_value + (i-3)*ms_diff]
#     isotop_arr.append(not filtered_raws_1.empty)
# isotop_arr

charge_ms_diff = {
    1: 1.003,
    2: 0.5015,
    3: 0.3343,
    4: 0.2508
}
precursor_charges = {}

for i in range(100):
    best_charge = None
    for key, val in charge_ms_diff.items():
        isotop_arr = findAdjecentPeakForThePrecursor( 
            index_location = i, 
            peaks_in_selected_precursors=peak_df, 
            precursorDf=sorted_selected_precursors,
            ms_diff= val )
        if (False in isotop_arr):
            print(key, val)
            print(isotop_arr)
        # if any(isotop_arr):
        #     print(key, val)
        #     print(isotop_arr)
        
        # print(isotop_arr)
        if False not in isotop_arr:  
            best_charge = key  # Assign charge if all isotopic peaks are found
            break  # Stop checking once a valid charge is found

    if best_charge:
        precursor_charges[i] = best_charge  # Store detected charge
        print(f"Precursor {i} has charge: {best_charge}")
    else:
        print(f"Precursor {i} has no valid charge detected.")
    

## consider the distrubution of the adjesent to peaks



In [ ]:
# filtered_raws_2 = selected_precursors[selected_precursors['mz_values'] == mz_value + charge_2_ms_diff ]
# filtered_raws_2
len(peak_df)

### Peak findings

In [ ]:
# peaks,_ = find_peaks(selected_precursors['intensity_values'], height=200)

# # peaks, _ = find_peaks(selected_precursors['intensity_values'], distance=20)
# peaks2, _ = find_peaks(selected_precursors['intensity_values'], prominence=1)
# # BEST!
# peaks3, _ = find_peaks(selected_precursors['intensity_values'], width=20)
# peaks4, _ = find_peaks(selected_precursors['intensity_values'], threshold=0.4) 

# len(selected_precursors['mz_values'].iloc[peaks])

In [ ]:
# plt.subplot(2, 2, 1)
# plt.plot(filtered_precursor.iloc[peaks]['intensity_values'], "xr"); 
# plt.plot(filtered_precursor['intensity_values']); plt.legend(['distance'])

# plt.subplot(2, 2, 2)
# plt.plot(peaks2, filtered_precursor.iloc[peaks2]['intensity_values'], "ob"); 
# plt.plot(filtered_precursor['intensity_values']); plt.legend(['prominence'])

# plt.subplot(2, 2, 3)
# plt.plot(peaks3, filtered_precursor.iloc[peaks3]['intensity_values'], "vg"); 
# plt.plot(filtered_precursor['intensity_values']); plt.legend(['width'])

# plt.subplot(2, 2, 4)
# plt.plot(peaks4, filtered_precursor.iloc[peaks4]['intensity_values'], "xk"); 
# plt.plot(filtered_precursor['intensity_values']); plt.legend(['threshold'])
# plt.show()

In [ ]:
sorted_indices = filtered_precursor['mz_values'].sort_values().index
sorted_mz_values = filtered_precursor.loc[sorted_indices, 'mz_values'].values
sorted_intensity_values = filtered_precursor.loc[sorted_indices, 'intensity_values'].values

# Plot the entire spectrum
plt.plot(sorted_mz_values, sorted_intensity_values, marker='o', label="Values")

# Plot the peaks in red
plt.plot(sorted_mz_values[peaks], sorted_intensity_values[peaks], color='red', marker='x', label="Peaks")


plt.legend()
plt.show()


In [ ]:
# from scipy.signal import savgol_filter

# # Example m/z and intensity data (Replace with real data)
# mz = np.linspace(100, 1000, 900)  # Simulated m/z values
# intensity = np.exp(-((mz - 400) ** 2) / (2 * 50 ** 2)) * 100 + \
#             np.exp(-((mz - 700) ** 2) / (2 * 30 ** 2)) * 80 + \
#             np.random.normal(0, 2, len(mz))  # Simulated peaks + noise



Resolution Type	Recommended Window Length


High-resolution MS (Orbitrap, FT-ICR)	7–15

Medium-resolution MS (TOF, Q-TOF)	11–21

Low-resolution MS (quadrupole, ion trap)	15–25

In [ ]:
sorted_indices = selected_precursors['mz_values'].sort_values().index
sorted_mz_values = selected_precursors.loc[sorted_indices, 'mz_values'].values
sorted_intensity_values = selected_precursors.loc[sorted_indices, 'intensity_values'].values

In [ ]:

# Apply Savitzky-Golay filter (smoothing + first derivative)
window_length = 11  # Choose based on resolution
polyorder = 3
first_derivative = savgol_filter(sorted_intensity_values, window_length, polyorder, deriv=1)

# Find zero crossings of the first derivative (peak detection)
zero_crossings = np.where(np.diff(np.sign(first_derivative)) < 0)[0]

# Filter peaks based on intensity threshold (to remove noise)
threshold = np.percentile(sorted_intensity_values, 75)  # Select peaks above 75th percentile
peaks = [i for i in zero_crossings if sorted_intensity_values[i] > threshold]

# Plot results
plt.figure(figsize=(10, 5))
plt.plot(sorted_mz_values, sorted_intensity_values, label="sorted_intensity_values (Raw Data)", color='black', alpha=0.6)
plt.scatter(sorted_mz_values[peaks], sorted_intensity_values[peaks], color='red', label="Detected Peaks", zorder=3)
plt.xlabel("m/z")
plt.ylabel("Intensity")
plt.title("Peak Detection Using Savitzky-Golay Derivatives")
plt.legend()
plt.show()


In [ ]:
original_peak_indices  = sorted_indices[peaks]
peaks_in_selected_precursors = selected_precursors.loc[original_peak_indices]
peaks_in_selected_precursors

In [ ]:
def group_isotopes_optimized(df, mz_tolerance=0.02, rt_tolerance=0.3, mobility_tolerance=0.01):
    min_values = df.min()
    max_values = df.max()
    
    rt_min_value = min_values['rt_values']
    rt_max_value = max_values['rt_values']
    
    mobility_min_value = min_values['mobility_values']
    mobility_max_value = max_values['mobility_values']
    
    mz_min_values = min_values['mz_values']
    mz_max_values = max_values['mz_values']
    
    
    df = df.sort_values(by=["rt_values"]).reset_index(drop=True)
    rt_bins = np.arange(rt_min_value, rt_max_value + rt_tolerance, rt_tolerance)
    df["rt_group"] = pd.cut(df["rt_values"], rt_bins,  labels=rt_bins[:-1])
    
    
    df = df.sort_values(by=["mobility_values"]).reset_index(drop=True)
    mobility_bins = np.arange(mobility_min_value, mobility_max_value + mobility_tolerance, mobility_tolerance)
    df["mobility_group"] = pd.cut(df["mobility_values"], mobility_bins,  labels=mobility_bins[:-1])
    
    
    df = df.sort_values(by=["mz_values"]).reset_index(drop=True)
    mobility_bins = np.arange(mz_min_values, mz_max_values +mz_tolerance, mz_tolerance)
    df["mz_group"] = pd.cut(df["mz_values"], mobility_bins,  labels=mobility_bins[:-1])
    
    

    # grouped_sets = set(df.groupby(["rt_group", "mobility_group", "mz_group"]).groups.keys())
    df["peptide_group"] = df.groupby(["rt_group", "mobility_group", "mz_group"]).ngroup()


    
    return df
    
    


# # Group isotopes (FAST!)
# isotope_groups = group_isotopes_optimized(window_1_precursor)

# # Print grouped isotope sets
# for idx, group in enumerate(isotope_groups):
#     print(f"Isotope Group {idx + 1}:")
#     print(group, "\n")


### Finding out the eg rt and mobility value to continue with the selection of similar Precursor and fragments to consider

To proceed with the selection of appropriate precursors and fragments, it is essential to determine the values of eg, rt, and mobility. These parameters will serve as the foundation for identifying and evaluating similar candidates, ensuring a systematic and informed approach to the selection process.

In [ ]:
window_1_precursor_rt = window_1_precursor['rt_values']
window_1_precursor_rt.value_counts()

import matplotlib.pyplot as plt

window_1_precursor_rt.value_counts().plot(kind="bar")
plt.xlabel("Categories")
plt.ylabel("Count")
plt.title("Bar Chart of Column Data")
plt.xticks(rotation=90)  # Rotate labels 90 degrees
# plt.yscale("log")
plt.show()

# here are the values couunt of rt values of all the precursors in the 1st quadrupole windows
window_1_precursor_rt.value_counts()

In [ ]:
window_1_precursor_mobility = window_1_precursor['mobility_values']
window_1_precursor_mobility.value_counts()

import matplotlib.pyplot as plt

window_1_precursor_mobility.value_counts().plot(kind="bar")
plt.xlabel("Categories")
plt.ylabel("Count")
plt.title("Bar Chart of Column Data")
plt.xticks(rotation=90)  # Rotate labels 90 degrees
# plt.yscale("log")
plt.show()

# here are the values couunt of rt values of all the precursors in the 1st quadrupole windows
window_1_precursor_mobility.value_counts()

### Extracting Fragments of the Window 1 precursors

In [ ]:
quad_low_mz_value = 400.0
selected_precursor_mobility = 0.765332
mobility_threshold = 0.06


selected_precursor_rt_value = 141.751394
rt_value_threshold = 60


precurser_index_1_fragments= data[
    selected_precursor_rt_value - rt_value_threshold: selected_precursor_rt_value + rt_value_threshold,
    selected_precursor_mobility - mobility_threshold: selected_precursor_mobility + mobility_threshold,
    1,
    :]

window_1_fragments = precurser_index_1_fragments[precurser_index_1_fragments["quad_low_mz_values"]== quad_low_mz_value]
# window_1_fragments.sort_values(by="intensity_values",ascending= False)
window_1_fragments

window_1_fragments = data[:,:,1,]

Selecting the fragment related to the quadrupole window

In [ ]:
window_1_fragments= data[:,:,1,:]   #data.__getitem__((slice(None), slice(None), 2, slice(None)))
quad_low_mz_values = window_1_fragments['quad_low_mz_values'].unique()
quad_high_mz_values = window_1_fragments['quad_high_mz_values'].unique()
print(quad_high_mz_values,quad_low_mz_values)

# i= 0
# window_1_precursor = window_1_precursor[(precursor_temp['mz_values']>=quad_low_mz_values[2-i]) & (precursor_temp['mz_values']<quad_high_mz_values[2-i])]
# window_1_precursor


In [ ]:
import numpy as np
import pandas as pd


# finding the similar raws with the selected thresholds 
def group_similar_rows(df, rt_tolerance, mobility_tolerance, mz_tolerance):
    min_values = df.min()
    max_values = df.max()
    
    rt_min_value = min_values['rt_values']
    rt_max_value = max_values['rt_values']
    
    mobility_min_value = min_values['mobility_values']
    mobility_max_value = max_values['mobility_values']
    
    mz_min_values = min_values['mz_values']
    mz_max_values = max_values['mz_values']
    
    
    df = df.sort_values(by=["rt_values"]).reset_index(drop=True)
    rt_bins = np.arange(rt_min_value, rt_max_value + rt_tolerance, rt_tolerance)
    df["rt_group"] = pd.cut(df["rt_values"], rt_bins,  labels=rt_bins[:-1])
    
    
    df = df.sort_values(by=["mobility_values"]).reset_index(drop=True)
    mobility_bins = np.arange(mobility_min_value, mobility_max_value + mobility_tolerance, mobility_tolerance)
    df["mobility_group"] = pd.cut(df["mobility_values"], mobility_bins,  labels=mobility_bins[:-1])
    
    
    df = df.sort_values(by=["mz_values"]).reset_index(drop=True)
    mobility_bins = np.arange(mz_min_values, mz_max_values +mz_tolerance, mz_tolerance)
    df["mz_group"] = pd.cut(df["mz_values"], mobility_bins,  labels=mobility_bins[:-1])
    
    

    grouped_df = df.groupby(["rt_group", "mobility_group", "mz_group"], observed=True).agg({
        "rt_values": "mean",
        "mobility_values": "mean",
        "mz_values": "mean",
        "intensity_values": "sum"
    }).reset_index(drop=True)
    
    


    return grouped_df

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

def group_isotopes_optimized(df, mz_tolerance=0.02, rt_tolerance=0.3, mobility_tolerance=0.01):
    min_values = df.min()
    max_values = df.max()
    
    rt_min_value = min_values['rt_values']
    rt_max_value = max_values['rt_values']
    
    mobility_min_value = min_values['mobility_values']
    mobility_max_value = max_values['mobility_values']
    
    mz_min_values = min_values['mz_values']
    mz_max_values = max_values['mz_values']
    
    
    df = df.sort_values(by=["rt_values"]).reset_index(drop=True)
    rt_bins = np.arange(rt_min_value, rt_max_value + rt_tolerance, rt_tolerance)
    df["rt_group"] = pd.cut(df["rt_values"], rt_bins,  labels=rt_bins[:-1])
    
    
    df = df.sort_values(by=["mobility_values"]).reset_index(drop=True)
    mobility_bins = np.arange(mobility_min_value, mobility_max_value + mobility_tolerance, mobility_tolerance)
    df["mobility_group"] = pd.cut(df["mobility_values"], mobility_bins,  labels=mobility_bins[:-1])
    
    
    df = df.sort_values(by=["mz_values"]).reset_index(drop=True)
    mobility_bins = np.arange(mz_min_values, mz_max_values +mz_tolerance, mz_tolerance)
    df["mz_group"] = pd.cut(df["mz_values"], mobility_bins,  labels=mobility_bins[:-1])
    
    

    # grouped_sets = set(df.groupby(["rt_group", "mobility_group", "mz_group"]).groups.keys())
    df["peptide_group"] = df.groupby(["rt_group", "mobility_group", "mz_group"]).ngroup()


    
    return df
    
    


# Group isotopes (FAST!)
isotope_groups = group_isotopes_optimized(window_1_precursor)

# # Print grouped isotope sets
# for idx, group in enumerate(isotope_groups):
#     print(f"Isotope Group {idx + 1}:")
#     print(group, "\n")


In [ ]:
# for idx, group in enumerate(isotope_groups):
#     print(f"Isotope Group {idx + 1}:")
#     print(group, "\n")
#     break
isotope_groups[isotope_groups['peptide_group'].isnull()]

In [ ]:
grouped_window_1_precursors =  group_similar_rows(
    window_1_precursor, 
    rt_tolerance= 60, 
    mobility_tolerance=0.06, 
    mz_tolerance= 0.1
    ).sort_values(by="intensity_values",ascending= False)


# this contains the peptide detector events in the selected quadropole window
# grouped_window_1_precursors.sort_values(by="mobility_values",ascending= False)
grouped_window_1_precursors.sort_values(by="mz_values")


In [ ]:
# These selected_precursor_mobility and selected_precursor_rt_value are needed
# to find the fragment detector events
quad_low_mz_value = 400.0
selected_precursor_mobility = 0.760861
mobility_threshold = 0.001


selected_precursor_rt_value = 115.976836
rt_value_threshold = 1


precurser_index_1_fragments= data[
    selected_precursor_rt_value - rt_value_threshold: selected_precursor_rt_value + rt_value_threshold,
    selected_precursor_mobility - mobility_threshold: selected_precursor_mobility + mobility_threshold,
    1,
    :]

window_1_fragments = precurser_index_1_fragments[precurser_index_1_fragments["quad_low_mz_values"]== quad_low_mz_value]
# window_1_fragments.sort_values(by="intensity_values",ascending= False)
window_1_fragments



In [ ]:
window_1_fragments[["mz_values","intensity_values"]].sort_values(by="mz_values")

In [ ]:
amino_acid_masses_acid_to_weight = {
    'Ala': 71,  # Alanine
    'Arg': 156,  # Arginine
    'Asn': 114,  # Asparagine
    'Asp': 115,  # Aspartic acid
    'Cys': 103,  # Cysteine
    'Gln': 128,  # Glutamine
    'Glu': 129,  # Glutamic acid
    'Gly': 57,   # Glycine
    'His': 137,  # Histidine
    'Ile': 113,  # Isoleucine
    'Leu': 113,  # Leucine
    'Lys': 128,  # Lysine
    'Met': 131,  # Methionine
    'Phe': 147,  # Phenylalanine
    'Pro': 97,   # Proline
    'Ser': 87,   # Serine
    'Thr': 101,  # Threonine
    'Trp': 186,  # Tryptophan
    'Tyr': 163,  # Tyrosine
    'Val': 99    # Valine
}


amino_acid_masses_weigth_to_acid = {
    57: 'Gly',   # Glycine
    71: 'Ala',   # Alanine
    87: 'Ser',   # Serine
    97: 'Pro',   # Proline
    99: 'Val',   # Valine
    101: 'Thr',  # Threonine
    103: 'Cys',  # Cysteine
    113: ['Leu','Ile'],  # Leucine, Isoleucine
    114: 'Asn',  # Asparagine
    115: 'Asp',  # Aspartic acid
    128: 'Lys',  # Lysine
    128: 'Gln',  # Glutamine
    129: 'Glu',  # Glutamic acid
    131: 'Met',  # Methionine
    137: 'His',  # Histidine
    147: 'Phe',  # Phenylalanine
    156: 'Arg',  # Arginine
    163: 'Tyr',  # Tyrosine
    186: 'Trp',  # Tryptophan
}


In [ ]:
# finding the precursor mass
precursor_mz = 413.053060
precursor_charge = 2
precursor_mass = precursor_mz * precursor_charge - precursor_charge

In [ ]:
def check_key_within_threshold(my_dict, key, threshold):
    for k in my_dict:
        if abs(k - key) <= threshold:
            return True
    
    return False


def get_values_within_threshold(my_dict, key, threshold):
    values_within_threshold = []
    
    # Iterate through all keys in the dictionary
    for k in my_dict:
        # Check if the absolute difference between key k and the given key is within the threshold
        if abs(k - key) <= threshold:
            # If within the threshold, append the corresponding value to the result list
            values_within_threshold.append(my_dict[k])
    
    return values_within_threshold


In [ ]:
# here we get the fragments whose mass is less than the precursor mass
# fragment charge is considered as 1.
# So mz values is equals to its mass

fragement_temp = window_1_fragments[["mz_values","intensity_values"]][window_1_fragments["mz_values"]<precursor_mass].sort_values(by="mz_values",ascending=False)

valid_fragment_indices ={
    # key = raw index, 
    # value  = 
}

for index, row in fragement_temp.iterrows():
    mass = row["mz_values"]
    intensity = row["intensity_values"]
    # print(f"Mass: {mass}, Intensity: {intensity}")
    
    
    y_ion_mass = mass
    b_ion_mass = precursor_mass - y_ion_mass
    mass_threshold = 1
    
    print(f"y_ion_mass: {y_ion_mass}, b_ion_mass: {b_ion_mass}")
    
    # if( b_ion_mass in amino_acid_masses_weigth_to_acid ): #need to add the threshold
    if( check_key_within_threshold(amino_acid_masses_weigth_to_acid, b_ion_mass, mass_threshold)): #need to add the threshold
        print(b_ion_mass, ' is in ')
        break
           

fragement_temp

In [ ]:
import numpy as np
import pandas as pd

quad_mz_min_value = 200
quad_mz_max_value = 1200


def quantize_mz_and_sum_intensity(df, mz_min, mz_max, bin_width):
    # Define m/z bin edges
    bins = np.arange(mz_min, mz_max + bin_width, bin_width)
    # print((bins))


    df["mz_bin"] = pd.cut(df["mz_values"], bins, right=False, labels=bins[:-1])
    # print(df.groupby("mz_bin", observed=True))

    # Compute weighted sum of intensities for each bin
    grouped_df = df.groupby("mz_bin")["intensity_values"].sum().reset_index()

    return grouped_df

bin_width = 5 # Define bin width

result_df = quantize_mz_and_sum_intensity(window_1_fragments, quad_mz_min_value, quad_mz_max_value, bin_width)
print(result_df)


grouping MS1 detector events by similar rows


In [ ]:
# import pandas as pd
# import numpy as np

# # Load data (assuming you already have a DataFrame named df)
# # Define similarity thresholds
# rt_tolerance = 1e-3  # Adjust based on precision needed
# mobility_tolerance = 1e-3
# mz_tolerance = 1e-2

# # Sort values to ensure similar ones are adjacent
# df = window_1_precursor.sort_values(by=["rt_values"])

# # Group similar rows
# def group_similar_rows(df):
#     grouped_data = []
#     current_group = [df.iloc[0]]

#     for i in range(1, len(df)):
#         prev = current_group[-1]
#         curr = df.iloc[i]

#         if (
#             abs(prev["rt_values"] - curr["rt_values"]) < rt_tolerance and
#             abs(prev["mobility_values"] - curr["mobility_values"]) < mobility_tolerance and
#             abs(prev["mz_values"] - curr["mz_values"]) < mz_tolerance
#         ):
#             current_group.append(curr)
#         else:
#             # Aggregate and save the group
#             merged_row = merge_rows(current_group)
#             grouped_data.append(merged_row)
#             current_group = [curr]

#     # Process last group
#     if current_group:
#         grouped_data.append(merge_rows(current_group))

#     return pd.DataFrame(grouped_data)

# # Function to merge a group of similar rows
# def merge_rows(group):
#     merged = group[0].copy()
#     merged["intensity_values"] = sum(row["intensity_values"] for row in group)
#     merged["corrected_intensity_values"] = sum(row["corrected_intensity_values"] for row in group)
#     return merged

# # Apply grouping
# filtered_df = group_similar_rows(df)

# # Display the final dataset
# print(filtered_df)


In [ ]:
# import numpy as np

# frame_indices = alphatims.bruker.convert_slice_key_to_int_array(data, slice(0,10000), "frame_indices")
# scan_indices = alphatims.bruker.convert_slice_key_to_int_array(data, slice(0,500000), "scan_indices")
# precursor_indices = np.array([[0, 1, 1]])
# tof_indices = alphatims.bruker.convert_slice_key_to_int_array(data, slice(0,500000), "tof_indices")
# quad_values = np.array([[-1, 0]])
# intensity_slices = np.arange(0, 1000)


# intensity_values = alphatims.bruker.convert_slice_key_to_float_array(
#                         slice(0,1000)
#                     )

            
# elected_indices = alphatims.bruker.filter_indices(
#                         frame_slices=frame_indices,
#                         scan_slices=scan_indices,
#                         precursor_slices= precursor_indices,
#                         tof_slices=tof_indices,
#                         quad_slices=quad_values,
#                         intensity_slices=intensity_values,
#                         frame_max_index=data.frame_max_index,
#                         scan_max_index=data.scan_max_index,
#                         push_indptr=data.push_indptr,
#                         precursor_indices=data.precursor_indices,
#                         quad_mz_values=data.quad_mz_values,
#                         quad_indptr=data.quad_indptr,
#                         tof_indices=data.tof_indices,
#                         intensities=data.intensity_values
#                     )

# elected_events = data.as_dataframe(
#     elected_indices
# )
# elected_events

Using np grouping

In [ ]:
# import pandas as pd

# data_range = 77042361 
# chunk_size = 7704236  # Number of rows per chunk




# # Iterate through the DataFrame in chunks
# for i in range(0, data_range, chunk_size):  # Use 'data_range' instead of 'range'
#     chunk = precursor_temp.iloc[i:i + chunk_size]
#     temp_df = chunk.groupby(
#         ["frame_indices", "scan_indices","precursor_indices","quad_low_mz_values"]
#         ).agg({
#         "raw_indices": list,
#         "push_indices": "first",
#         "tof_indices": "first",
#         "rt_values": "first",
#         "rt_values_min": "first",
#         "mobility_values": list,
#         "quad_high_mz_values": "first",
#         "mz_values": "first",
#         "intensity_values": "sum"
#     }).reset_index()
#     print(temp_df)
    

In [ ]:

    
# final_df = precursor_temp.head(100).groupby(
#     ["frame_indices", "scan_indices","precursor_indices","quad_low_mz_values"]
#     ).agg({
#     "raw_indices": list,
#     "push_indices": "first",
#     "tof_indices": "first",
#     "rt_values": "first",
#     "rt_values_min": "first",
#     "mobility_values": "first",
#     "quad_high_mz_values": "first",
#     "mz_values": "first",
#     "intensity_values": "sum"
# }).reset_index()

# # final_df
# # Merge back with the original DataFrame to keep all columns (other fields)
# final_df = pd.merge(precursor_temp, final_df, on=["frame_indices", "scan_indices","precursor_indices","quad_low_mz_values"], how='left')

In [ ]:
# total = 0
# for i in range(len(final_df["raw_indices"])):
#     total += len(final_df['raw_indices'][i])
    
# total

In [ ]:
# precursor_temp["intensity_values"]

In [ ]:
# precursor_temp.head()

In [ ]:
# precursor_temp['frame_indices'].unique()

In [ ]:

# # List to store new rows
# new_rows = []

# # Process each group
# for _, group in grouped:
#     # Check if mz_values are within the threshold
#     mz_values = group['mz_values'].values
#     intensity_values = group['intensity_values'].values

#     # If mz_values are within the threshold, combine them
#     if np.ptp(mz_values) <= mz_threshold:  # ptp is peak-to-peak (max - min)
#         # Linear interpolation to get a representative m/z value (e.g., the average)
#         interpolated_mz = np.mean(mz_values)
        
#         # Sum the intensities
#         total_intensity = np.sum(intensity_values)
        
#         # Create a new row with combined information
#         new_row = {
#             'frame_indices': group['frame_indices'].iloc[0],
#             'scan_indices': group['scan_indices'].iloc[0],
#             'precursor_indices': group['precursor_indices'].iloc[0],
#             'quad_low_mz_values': group['quad_low_mz_values'].iloc[0],
#             'mz_values': interpolated_mz,  # representative m/z value
#             'intensity_values': total_intensity  # summed intensity
#         }
        
#         # Append the new row to the list
#         new_rows.append(new_row)

# # Create a DataFrame from the new rows
# final_df = pd.DataFrame(new_rows)

# print(final_df)


In [ ]:
def inspect_peptide(
    dia_data,
    peptide,
    ppm=50,
    rt_tolerance=30, #seconds
    mobility_tolerance=0.05, #1/k0
    heatmap=False
):
    precursor_mz = peptide["mz"]
    precursor_mobility = peptide["mobility"]
    precursor_rt = peptide["rt"]
    fragment_mzs = peptide["fragment_mzs"]
    rt_slice = slice(
        precursor_rt - rt_tolerance,
        precursor_rt + rt_tolerance
    )
    im_slice = slice(
        precursor_mobility - mobility_tolerance,
        precursor_mobility + mobility_tolerance
    )
    precursor_mz_slice = slice(
        precursor_mz / (1 + ppm / 10**6),
        precursor_mz * (1 + ppm / 10**6)
    )
    precursor_indices = dia_data[
        rt_slice,
        im_slice,
        0, #index 0 means that the quadrupole is not used
        precursor_mz_slice,
        "raw"
    ]
    if heatmap:
        precursor_heatmap = alphatims.plotting.heatmap(
            dia_data.as_dataframe(precursor_indices),
            x_axis_label="rt",
            y_axis_label="mobility",
            title="precursor",
            width=250,
            height=250
        )
        overlay = precursor_heatmap
    else:
        precursor_xic = alphatims.plotting.line_plot(
            dia_data,
            precursor_indices,
            x_axis_label="rt",
            width=900,
            remove_zeros=True,
            label="precursor"
        )
        overlay = precursor_xic
    for fragment_name, mz in fragment_mzs.items():
        fragment_mz_slice = slice(
            mz / (1 + ppm / 10**6),
            mz * (1 + ppm / 10**6)
        )
        fragment_indices = dia_data[
            rt_slice,
            im_slice,
            precursor_mz_slice,
            fragment_mz_slice,
            "raw"
        ]
        if len(fragment_indices) > 0:
            if heatmap:
                fragment_heatmap = alphatims.plotting.heatmap(
                    dia_data.as_dataframe(fragment_indices),
                    x_axis_label="rt",
                    y_axis_label="mobility",
                    title=f"{fragment_name}: {mz:.3f}",
                    width=250,
                    height=250,
                )
                overlay += fragment_heatmap
            else:
                fragment_xic = alphatims.plotting.line_plot(
                    dia_data,
                    fragment_indices,
                    x_axis_label="rt",
                    width=900,
                    remove_zeros=True,
                    label=fragment_name,
                )
                overlay *= fragment_xic.opts(muted=True)
    if not heatmap:
        overlay.opts(hv.opts.Overlay(legend_position='bottom'))
        overlay.opts(hv.opts.Overlay(click_policy='mute'))
        overlay = overlay.opts(show_legend=True)
        hv.save(overlay, "tutorial_dia_xic_overlay.html")
    return overlay.opts(
        title=f"{peptide['sequence']}_{peptide['charge']}"
    )